# Minimal RAG for FDA events related to a stock

This notebook:
1. Loads FDA-approved drug data (March 2026)
2. Converts company names to tickers
3. Builds one FDA event document per row and uses embeddings to retrieve relevant FDA events
4. Adds a simple stock move feature using close minus open on the approval date
5. Optionally uses a Groq-hosted LLM (free) to answer with both FDA and stock-move context

## Key steps
1. Install packages (Cell 2)
2. Set imports and config (Cell 3)
3. Set `GROQ_API_KEY` (Cell 4)
4. Run Step 1-5 cells to load data and build RAG logic
5. Ask questions using the final cell

## Setup. Install packages
Install the Python packages used by this notebook.

In [1]:
%pip install -q pandas groq scikit-learn yfinance "sentence-transformers==2.7.0"

Note: you may need to restart the kernel to use updated packages.


## Setup. Imports and config
Load libraries and set file paths.

In [2]:
from pathlib import Path
import os
import pandas as pd
import yfinance as yf
from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity



/Users/xiaolei/opt/anaconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATA_FILE = Path("DrugsFDA_FDA-Approved_Drugs_March2026.csv")

# Set API key in terminal: %env GROQ_API_KEY=your-key
# Or: export GROQ_API_KEY="your-key" (local)

## Step 1. Load FDA data
Read the FDA approval CSV into a dataframe.

In [4]:
# 1. Load FDA calendar data
## As a quick prototype, the data is downloaded manually from the DrugFDA https://www.accessdata.fda.gov/scripts/cder/daf/index.cfm
if not DATA_FILE.exists():
    raise FileNotFoundError(f"FDA data file not found: {DATA_FILE}")

df = pd.read_csv(DATA_FILE)

df["Approval Date"] = pd.to_datetime(df["Approval Date"], errors="coerce")

df.head(n=5)

,Approval Date,Drug Name,Submission,Active Ingredients,Company,Submission Classification *,Submission Status
0,2026-03-02,IMODIUM MULTI-SYMPTOM RELIEFNDA #021140,SUPPL-39,LOPERAMIDE HYDROCHLORIDE; SIMETHICONE,KENVUE BRANDS,Manufacturing (CMC),Approval
1,2026-03-02,METRONIDAZOLEANDA #070772,SUPPL-50,METRONIDAZOLE,INNOGENIX,Labeling,Approval
2,2026-03-02,ONDANSETRON HYDROCHLORIDEANDA #078539,SUPPL-16,ONDANSETRON HYDROCHLORIDE,AUROBINDO PHARMA,Labeling,Approval
3,2026-03-02,BOSULIFNDA #203341,SUPPL-27,BOSUTINIB MONOHYDRATE,PF PRISM CV,Labeling,Approval
4,2026-03-02,MOXIFLOXACIN HYDROCHLORIDEANDA #209143,ORIG-1,MOXIFLOXACIN HYDROCHLORIDE,MACLEODS PHARMS LTD,NaN,Approval


## Step 2. Resolve tickers
Create the helper functions used to map company names to stock tickers.

In [5]:
# Resolve company names to stock tickers
MANUAL_TICKER_MAP = {
    "KENVUE": "KVUE", "KENVUE BRANDS": "KVUE",
    "JANSSEN BIOTECH": "JNJ", "BMS": "BMY",
    "BRISTOL MYERS SQUIBB": "BMY", "ELI LILLY": "LLY",
    "MERCK": "MRK",
}

def resolve_ticker(company_name: str):
    if pd.isna(company_name):
        return None
    
    raw = str(company_name).strip().upper()
    if raw in MANUAL_TICKER_MAP:
        return MANUAL_TICKER_MAP[raw]
    
    # Try Yahoo Finance fuzzy search
    try:
        search = yf.Search(query=raw, max_results=3, news_count=0, enable_fuzzy_query=True)
        if search.quotes:
            return search.quotes[0].get("symbol")
    except:
        pass
    return None

## Step 3. Build ticker column
Apply the ticker mapping to the FDA dataframe.

In [6]:
df["ticker"] = df["Company"].apply(resolve_ticker)
df[["Approval Date", "Company", "Drug Name", "ticker"]].head()

,Approval Date,Company,Drug Name,ticker
0,2026-03-02,KENVUE BRANDS,IMODIUM MULTI-SYMPTOM RELIEFNDA #021140,KVUE
1,2026-03-02,INNOGENIX,METRONIDAZOLEANDA #070772,2591.HK
2,2026-03-02,AUROBINDO PHARMA,ONDANSETRON HYDROCHLORIDEANDA #078539,AUROPHARMA.BO
3,2026-03-02,PF PRISM CV,BOSULIFNDA #203341,CVEBRX=X
4,2026-03-02,MACLEODS PHARMS LTD,MOXIFLOXACIN HYDROCHLORIDEANDA #209143,ZIM


## Step 4. Build RAG
Create embeddings, retrieve FDA events, and fetch stock prices.

In [8]:
# Build embeddings indexed by FDA events
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create event text from each FDA approval row
def make_event_text(row):
    return f"date: {row['Approval Date']} | company: {row['Company']} | ticker: {row['ticker']} | drug: {row['Drug Name']} | ingredients: {row['Active Ingredients']}"

df["event_text"] = df.apply(make_event_text, axis=1)
doc_matrix = embedding_model.encode(df["event_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)

# Fetch stock price change on approval date
def get_price_change(ticker, approval_date):
    if not ticker or pd.isna(approval_date):
        return pd.NA, pd.NA, pd.NA
    try:
        start = pd.Timestamp(approval_date).date()
        price_df = yf.download(str(ticker), start=start, end=start+pd.Timedelta(days=1), 
                               auto_adjust=False, progress=False, threads=False)
        if price_df.empty:
            return pd.NA, pd.NA, pd.NA
        if isinstance(price_df.columns, pd.MultiIndex):
            price_df.columns = price_df.columns.get_level_values(0)
        if "Open" in price_df.columns and "Close" in price_df.columns:
            o = float(price_df.iloc[0]["Open"])
            c = float(price_df.iloc[0]["Close"])
            return o, c, c - o
    except:
        pass
    return pd.NA, pd.NA, pd.NA

# Retrieve FDA events for a question
def ask_question(question: str, stock: str | None = None):
    # Filter by stock if provided
    if stock:
        mask = df["ticker"].str.upper() == stock.upper()
        search_df = df[mask]
    else:
        search_df = df
    
    # Encode question and find matches
    q_vec = embedding_model.encode([question], normalize_embeddings=True, show_progress_bar=False)
    scores = cosine_similarity(q_vec, doc_matrix[search_df.index]).ravel()
    top_idx = search_df.index[scores.argsort()[::-1][:5]]
    
    hits = df.iloc[top_idx].copy()
    hits["Price Change"] = hits.apply(lambda r: get_price_change(r["ticker"], r["Approval Date"])[2], axis=1)
    
    # Build context for LLM
    context = "\n".join([
        f"{r['Approval Date'].date()} | {r['Company']} | {r['Drug Name']} | Status: {r['Submission Status']} | Price Change: {r['Price Change']:.2f}%" 
        if pd.notna(r['Price Change']) else f"{r['Approval Date'].date()} | {r['Company']} | {r['Drug Name']} | Status: {r['Submission Status']}"
        for _, r in hits.iterrows()
    ])
    
    # Call LLM
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        return f"Retrieved {len(hits)} FDA events:\n{context}", hits
    
    client = Groq(api_key=api_key)
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{
            "role": "system",
            "content": "You are a financial assistant. Answer questions about the impact of FDA events on stock price changes on approval dates (close - open). Don't use bullet point. If there are multiple events, try to make a concise summary. "
        }, {
            "role": "user",
            "content": f"Question: {question}\n\nFDA Events:\n{context}"
        }],
        temperature=0.1,
    )
    return response.choices[0].message.content.strip(), hits



/Users/xiaolei/opt/anaconda3/envs/py311/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Step 5. Run an example
Ask one example question and inspect the supporting rows.

In [9]:
answer, results = ask_question("What are the latest FDA events related to Johnson and Johnson and how did the stock move on those approval dates?", stock="JNJ")
print(answer)
print("\n" + "="*80)
display(results[["Approval Date", "Company", "ticker", "Drug Name", "Submission Status", "Price Change"]])

On March 5, 2026, Johnson and Johnson's subsidiary, Janssen Biotech, received FDA approvals for three new treatments: Erleadanda, Talveybla, and Tecvaylibla. The stock price of Johnson and Johnson moved downward on the approval dates, with a price change of -3.00% for each of the approved treatments. This suggests that the market may have been expecting more significant or positive outcomes from these approvals, leading to a slight decline in the stock price.



,Approval Date,Company,ticker,Drug Name,Submission Status,Price Change
44,2026-03-05,JANSSEN BIOTECH,JNJ,ERLEADANDA #210951,Approval,-3.0
52,2026-03-05,JANSSEN BIOTECH,JNJ,TALVEYBLA #761342,Approval,-3.0
51,2026-03-05,JANSSEN BIOTECH,JNJ,TECVAYLIBLA #761291,Approval,-3.0
